# Сквозной RAG-проект — курс LLM-инженер

Один корпус, один золотой набор, серия экспериментов по пяти контрольным точкам:

| КТ | Тема | Статус |
|---|---|---|
| 1 | Корпус и золотой набор | **готово** (раздел ниже, код в `src/`) |
| 2 | Предобработка, чанкинг и метаданные | план (§2) |
| 3 | Векторная БД, эмбеддинги, sparse/dense/hybrid | план (§3) |
| 4 | Трансформация запросов, реранкинг | план (§4) |
| 5 | Генерация, оценка качества, продвинутый RAG | план (§5) |

Источники: `data/corpus.jsonl` (сырой wikitext вики The Binding of Isaac: Rebirth),
`data/gold/gold_full.json` (60 вопросов). Отчёт по КТ 1: `reports/kt1/otchet_kt1.md`.
Запуск: из папки `notebooks/`.

## 1. КТ 1 — корпус и золотой набор

### 1.1 Окружение

Корпус: вся вики **The Binding of Isaac: Rebirth** (`bindingofisaacrebirth.wiki.gg`) —
«вики-подобный ресурс с шаблонами, разметкой и служебными страницами» (вариант из ТЗ).
Выгрузка — `src/fetch_wiki.py` (asyncio + aiohttp, 20 параллельных): 3 439 страниц
namespace 0 через MediaWiki API, wikitext батчами по 50. Сохранён **сырой wikitext**
(шаблоны `{{...}}`, ссылки `[[...]]`, infobox'ы) — «грязный» источник по требованию ТЗ.
Отброшено 1 661 redirect-страница (`#REDIRECT [[...]]`) и страницы <40 символов.

In [ ]:
import sys
from pathlib import Path

import json

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))  # notebook живёт в notebooks/

from common import (BASELINE_OUT, GOLD_FULL, GOLD_TEST, GOLD_VAL,
                    load_corpus, with_tokens)

### 1.2 Корпус: объём и длины

Минимумы ТЗ — 500 документов и 200k токенов.

In [ ]:
corpus = with_tokens(load_corpus())
pd.Series({
    "документов": len(corpus),
    "токенов суммарно": int(corpus.n_tokens.sum()),
    "средняя длина, токенов": round(corpus.n_tokens.mean()),
    "медиана": int(corpus.n_tokens.median()),
    "очень короткие (<50)": int((corpus.n_tokens < 50).sum()),
    "очень длинные (>2000)": int((corpus.n_tokens > 2000).sum()),
}).to_frame().T

### 1.3 EDA

Обязательное по ТЗ: (1) распределение длин — гистограмма; (2) доля «мусорных» токенов
(эмодзи, URL, спецсимволы, числа); (3) дубликаты и near-duplicates (MinHash) — зафиксировать,
**не удалять**. Логика — `src/eda.py`; результат — `reports/kt1/lengths_hist.png`.

In [ ]:
import eda

summary = eda.run()  # напечатает полный summary и сохранит гистограмму
print(f"мусор: {summary['junk_share']}; near-dup: {summary['near_duplicate_docs']}")

In [ ]:
from IPython.display import Image, display

display(pd.Series({
    "эмодзи": f"{summary['junk_share']['emoji']:.2%}",
    "URL": f"{summary['junk_share']['url']:.2%}",
    "числа": f"{summary['junk_share']['num']:.2%}",
    "знаки препинания": f"{summary['junk_share']['punct']:.2%}",
    "exact-дубли": str(summary["exact_duplicates"]),
    "near-dup (Jaccard>=0.7)": str(summary["near_duplicate_docs"]),
    "доков с >20% мусора": str(summary["docs_gt20pct_junk"]),
}).to_frame().T)
Image(filename="../reports/kt1/lengths_hist.png", width=800)

**Выводы EDA** (полные — в отчёте): типичный документ ~700 токенов (медиана), длинный хвост —
большинство страниц-справок; основной шум — wikitext-разметка и числа (id, HP), эмодзи/URL
почти нет; ~84 near-dup (Jaccard ≥ 0.7) — зафиксированы для дедупликации на КТ 2. Для чанкинга
это значит: фиксированный чанкер 512 разрежет infobox'ы пополам; для ретрива — id/HP в тексте
будут создавать лексические ложные совпадения.

### 1.4 Золотой набор

60 пар «вопрос → релевантные документы» (ТЗ: 40–60). Схема записи:
`id` (1–60), `question` (EN, короткие и развёрнутые), `type`, `doc_ids` (страницы корпуса,
префикс `wiki://`; для негативных/абстрактных — пусто), `answer_contexts` (вербальные цитаты
wikitext ≤250 символов; для негативных/абстрактных — пусто).

Провенанс: генерация моделью по `prompts/gold_prompt.md` → автоверификация `build_gold_v2.py`
(цитаты assert'ятся подстрокой wikitext, doc_ids — точными заголовками, негативы — grep'ом по
корпусу; 2 негатива отклонены как «есть в корпусе») → ручная правка 3 abstract (id 33/38/40) →
добавлены 11 long-form вопросов (id 52–63).

In [ ]:
gold = json.load(open(GOLD_FULL, encoding="utf-8"))
gdf = pd.DataFrame(gold)
gdf.type.value_counts().reindex(["factual", "composite", "abstract", "negative"]).rename_axis("type")

In [ ]:
for t in ["factual", "composite", "abstract", "negative"]:
    ex = gdf[gdf.type == t].iloc[-1]
    print(f"--- {t} (id {ex.id}) ---")
    print("Q:", ex.question)
    print("doc_ids:", ex.doc_ids)
    for c in ex.answer_contexts[:2]:
        print("ctx:", c[:100].replace("\n", " "))
    print()

### 1.5 Разделение на validation / test

`src/gold.py split`: 70/30, стратификация по типу, `random_state=42`. Val — для подбора
конфигов (КТ 2–4), test — только для финальной оценки. Баланс сохраняется в обеих частях:
все типы присутствуют.

In [ ]:
val = pd.DataFrame(json.load(open(GOLD_VAL, encoding="utf-8")))
test = pd.DataFrame(json.load(open(GOLD_TEST, encoding="utf-8")))
pd.DataFrame({
    "validation": val.type.value_counts(),
    "test": test.type.value_counts(),
}).fillna(0).astype(int)

### 1.6 Baseline: LLM без RAG

Ответы модели **без добавленных контекстов** на test-вопросы (локальная LLM
Qwen3.8-27B-NVFP4; промпт: «отвечай кратко, если не знаешь — скажи „не знаю“»).
Регенерация: `python src/baseline.py` (адрес/ключ/модель — в `.env`).

In [ ]:
base = pd.DataFrame(json.load(open(BASELINE_OUT, encoding="utf-8")))
base["admit"] = base.answer.str.lower().str.contains("не знаю")
print(f"«не знаю»: {base.admit.sum()}/{len(base)}; средний latency: {base.latency_s.mean():.1f} c")
base.groupby("type").admit.agg(["sum", "count"]).rename(columns={"sum": "не_знаю", "count": "всего"})

In [ ]:
for _, r in base[~base.admit].head(4).iterrows():
    print(f"id={r.id} ({r.type}) Q: {r.question[:70]}")
    print("   A:", " ".join(str(r.answer).split())[:160])
    print()

**Вывод КТ 1.** Корпус «грязный» и объёмный (2.19M токенов), gold-набор покрывает все 4 типа
вопросов с вербальным ground truth. Модель без RAG: часть вопросов знает по параметрическим
знаниям об игре, но ответы неисточны и нестабильны (varies от прогона), long-tail факты (id,
HP, тайм-лимиты, составы пулов) — «не знаю». **RAG по корпусу необходим.** Следующий шаг —
КТ 2: предобработка, чанкинг, метаданные.

## 2. КТ 2 — предобработка, чанкинг и метаданные
Метрики — на **validation** split (42 вопроса, 26 с doc_ids, multi-doc: 1–3 релевантныхдокумента на вопрос; hit@k — хотя бы один релевантный doc в топ-k, recall@k — долярелевантных docs в топ-k); test не используется до финала. Ретрив-базовая линия: BM25.Все артефакты — `data/kt2/`, отчёт — `reports/kt2/otchet_kt2.md`.

In [ ]:
import kt2_retrieve as kr  # sys.path уже содержит src/
import json as _json

prep = _json.load(open("../data/kt2/prep_stats.json", encoding="utf-8"))
pd.DataFrame(prep).T
# строки: s0_raw -> s4_dedup; колонки: docs, junk_share (+ для s4: removed)

### 2.1 Baseline до чистки и таблица «до/после» каждого шага

`s0_raw + TokenChunker(512) + BM25` — точка отсчёта. Затем каждый шаг чистки отдельно
(каждый — файл `corpus_<stage>.jsonl`):

In [ ]:
stages = ["s0_raw", "s1_markup", "s2_links", "s3_normalize", "s4_dedup"]
# метрики сохранены в отчёте; пересчитать любой шаг: python src/kt2_retrieve.py eval <stage> token
prep_table = pd.DataFrame({
    "docs": [(prep[s]["docs"] if "docs" in prep[s] else prep[s]["after"]) for s in stages],
    "junk": [f"{prep[s]['junk_share']:.1%}" for s in stages],
    "removed": {**{s: 0 for s in stages[:4]}, "s4_dedup": prep["s4_dedup"]["exact_removed"] + prep["s4_dedup"]["near_removed"]},
}, index=stages)
prep_table

### 2.2 Чанкеры (chonkie) на s4_dedup + BM25

Четыре чанкера разных принципов; метрики и статистика длин (hit/recall @5/@10 на val):

In [ ]:
rows = []
for strat in ["token", "sentence", "semantic", "neural"]:
    ck = kr.make_chunks("s4_dedup", strat)  # кэш: data/kt2/chunks_s4_dedup_<strat>.jsonl
    rows.append({"chunker": strat, "chunks": len(ck),
                 "mean_tok": int(ck.n_tokens.mean()), "med_tok": int(ck.n_tokens.median()),
                 "std_tok": int(ck.n_tokens.std()),
                 "share<50": round(float((ck.n_tokens < 50).mean()), 3),
                 "share>500": round(float((ck.n_tokens > 500).mean()), 3)})
pd.DataFrame(rows)

### 2.3 Ablation SemanticChunker: пороги × 2 эмбеддера

Оптимум — **thr 0.5–0.6**; выбор эмбеддера границ почти не влияет (≤0.04).
Визуализация: `reports/kt2/sem_example.md` (страница «Mom» при порогах 0.4/0.6/0.8).

In [ ]:
sem = _json.load(open("../data/kt2/sem_ablation.json", encoding="utf-8"))
sem_df = pd.DataFrame(sem)
sem_df.pivot(index="model", columns="thr", values="recall@5").round(3)

### 2.4 Parent/Child (BGE-M3, fp16/CUDA)

Родители TokenChunker(p) → дети TokenChunker(c); индексируются дети, top-k детей →
родитель. Сравнение plain (дети напрямую) vs parent/child; ниже — по типам вопросов.

In [ ]:
pc = _json.load(open("../data/kt2/parent_child.json", encoding="utf-8"))
pd.DataFrame(pc).set_index(["parent", "child"])

### 2.5 Метаданные: 2 сценария применения

noise_score (доля неалфавитных токенов), top-5 TF-IDF, структура. Фильтр по noise —
нейтрален (шумных чанков 2.25% и их нет в топах); **prepend «Title. Keywords: …» —
+recall@10**.

In [ ]:
md_ = _json.load(open("../data/kt2/metadata.json", encoding="utf-8"))
pd.DataFrame({
    "baseline": {k: v for k, v in md_["A_filter"]["before"].items()},
    "A: фильтр noise<=0.5": {k: v for k, v in md_["A_filter"]["after"].items()},
    "B: prepend метаданных": {k: v for k, v in md_["B_prepend"]["after"].items()},
})

### 2.6 LLM-обработка чанков: суммаризация и HyDE

Exploratory-выборка (ТЗ допускает): 26 документов gold_val; остальной корпус — как есть.
Модель — Qwen3.8-27B-NVFP4 (локальная), kэш по sha1 чанка, стоимость — в JSON.

In [ ]:
llm = _json.load(open("../data/kt2/llm_chunks.json", encoding="utf-8"))
pd.DataFrame(llm["metrics_val"]).T
# llm["llm_total_tokens"], llm["summarize_wall_secs"], llm["hyde_wall_secs"]

**Вывод КТ 2 (multi-doc gold).** Чистка s0→s4 — основной выигрыш: recall@5 0.590→0.712(+0.122), recall@10 0.660→0.808 (+0.148); главный шаг — s1 (разметка), второй — s3(нормализация). TokenChunker 512 — лучший простой чанкер по всем метрикам (0.923/0.712/0.962/0.808). Semantic-чанкинг не дотягивает (лучшая точка 0.885/0.647), оптимальный порогзависит от эмбеддера. BGE-M3 dense впереди BM25 по всем четырём метрикам (0.962/0.737/1.0/0.859). Prepend метаданных на BM25 нейтрален (hit@5 −0.039) — перепроверяется на denseв КТ 3. LLM-методы (summary, HyDE) на BM25 теряют метрику — пересмотр на dense в КТ 3.Финальная фиксация для КТ 3 — в отчёте.

## 3. КТ 3 — векторная БД, эмбеддинги, sparse/dense/hybrid

Фиксируем baseline из КТ 2 (чанкер + предобработка) — пересмотрен не будет.

In [ ]:
# TODO 3.1 — выбрать БД (Milvus / Qdrant / ChromaDB), 2-3 конфигурации индекса; для каждой:
# время индексирования всего корпуса, медианное время запроса (100 запросов), пиковый RAM,
# Recall@5 на gold_val.

In [ ]:
# TODO 3.2 — scalability: проиндексировать 10%/25%/50%/100% корпуса, график «время запроса /
# память от размера корпуса».

In [ ]:
# TODO 3.3 — 3 эмбеддинг-модели разного размера (MTEB RU leaderboard: лёгкая/средняя/тяжёлая).
# Таблица: размер (МБ), время эмбеддинга корпуса и запроса, метрики на gold_val, MTEB score.
# Ответить: коррелирует ли позиция в MTEB с качеством на этом корпусе?

In [ ]:
# TODO 3.4 — три пайплайна: sparse (BM25), dense (эмбеддинги), hybrid (RRF/взвешенный).
# Общие метрики на gold: Precision@k, Recall@k, NDCG@k, k in {1, 5, 10}; отдельно val и test.

## 4. КТ 4 — трансформация запросов, реранкинг

In [ ]:
# TODO 4.1 — трансформация запроса: переписывание, HyDE, multi-query; каждый вариант —
# метрики на gold_val (выбор по val, финал на test).

In [ ]:
# TODO 4.2 — реранкинг: кандидатный пул, порог/к-реранкера; таблица до/после на val и test.

## 5. КТ 5 — генерация, оценка качества, продвинутый RAG

In [ ]:
# TODO 5.1 — генерация ответа по top-k ретриву; оценка: фактическая точность (LLM-as-judge /
# сверка с answer_contexts), faithfulness, «не знаю» на негативных вопросах.

In [ ]:
# TODO 5.2 — продвинутый RAG: corrective/self-RAG или цитирование источников в ответе;
# сравнение с базовым пайплайном.

In [ ]:
# TODO 5.3 — итоговая таблица: метрики validation и test по финальной конфигурации (обязательно
# раздельно, cf. ТЗ КТ 1: «в итоговой таблице отдельно показывать метрики validation и test»).

## Артефакты

| Файл | Что |
|---|---|
| `data/corpus.jsonl` | корпус: 1 776 доков, 2.19M токенов (сырой wikitext) |
| `data/gold/gold_full.json` | gold: 60 вопросов (источник) |
| `data/gold/gold_val.json` / `gold_test.json` | split 42/18 |
| `data/gold/baseline_no_rag.json` | ответы LLM без RAG (id/type/question/answer/latency_s) |
| `reports/kt1/otchet_kt1.md` | отчёт КТ 1 |
| `src/{common,fetch_wiki,eda,gold,baseline,run_kt1}.py` | пайплайн КТ 1 |